# SiteX: CSV to SpatiaLite Migration Pipeline

This notebook manages the transition from independent, unindexed CSV files to a unified, spatially-indexed SQLite database (`sitex_geospatial.db`). This improves data extraction speed exponentially and removes the O(n*m) complexity in feature engineering.

### Requirements:
Ensure you have the required packages installed in your `.venv`:
```bash
pip install pandas spatialite
```

In [9]:
%pip install pandas spatialite


Note: you may need to restart the kernel to use updated packages.


In [10]:
import sqlite3
import pandas as pd
import os
import sys
from pathlib import Path

# Paths configuration relative to backend directory
WORKSPACE_DIR = Path('..')
DB_PATH = WORKSPACE_DIR / 'sitex_geospatial.db'
CSV_DIR = WORKSPACE_DIR / 'DataEngineering' / 'CSV_Reference'

# Mapping CSV files to POI type
FILE_MAPPINGS = {
    'banks.csv': 'bank',
    'education.csv': 'school',
    'health.csv': 'hospital',
    'temples.csv': 'temple',
    'other.csv': 'other'
}

print(f"Python executable: {sys.executable}")
print(f"Database will be created at: {DB_PATH.resolve()}")
print(f"Looking for CSVs in: {CSV_DIR.resolve()}")
print(f"Is 64-bit Python? {sys.maxsize > 2**32}")

Python executable: /mnt/windows/Users/taman/projects/SiteX/.venv/bin/python
Database will be created at: /mnt/windows/Users/taman/projects/SiteX/backend/sitex_geospatial.db
Looking for CSVs in: /mnt/windows/Users/taman/projects/SiteX/backend/DataEngineering/CSV_Reference
Is 64-bit Python? True


## 1. Database Initialization
Create an empty database and enable the SpatiaLite extension which provides functions like `ST_Distance` and R-tree clustering indexing.

In [11]:
# Start fresh
if DB_PATH.exists():
    try:
        os.remove(DB_PATH)
        print("Removed exiting database for a clean start.")
    except PermissionError:
        print("Database is currently locked. Ensure no other applications are using it.")

# Establish connection
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Platform-specific setup for SpatiaLite
import platform
import sys

is_windows = platform.system() == 'Windows'
is_linux = platform.system() == 'Linux'

if is_windows:
    # Windows: Add mod_spatialite to the system PATH if it exists
    spatialite_dir = WORKSPACE_DIR / 'mod_spatialite-5.1.0-win-amd64'
    if spatialite_dir.exists():
        os.environ['PATH'] = str(spatialite_dir.resolve()) + ';' + os.environ.get('PATH', '')
        if hasattr(os, 'add_dll_directory'):
            os.add_dll_directory(str(spatialite_dir.resolve()))
        print(f"✓ Added local SpatiaLite directory to PATH: {spatialite_dir}")
    else:
        print(f"⚠️ SpatiaLite directory not found at {spatialite_dir}, will try system installation.")
elif is_linux:
    print("✓ Linux detected - using system-installed SpatiaLite")

# Enable spatial capabilities
conn.enable_load_extension(True)
try:
    conn.load_extension('mod_spatialite') 
    print("✓ SpatiaLite extension enabled.")
except Exception as e:
    print(f"⚠️ Error loading SpatiaLite: {e}")
    if is_linux:
        print("  On Linux, install SpatiaLite with: sudo apt-get install libspatialite-dev")
    raise e
conn.enable_load_extension(False)

# Initialize standard spatial metadata
cursor.execute("SELECT InitSpatialMetaData(1)")
conn.commit()
print("✓ Spatial metadata initialized.")

Removed exiting database for a clean start.
✓ Linux detected - using system-installed SpatiaLite
✓ SpatiaLite extension enabled.
✓ Spatial metadata initialized.


## 2. Defining Schema with Spatial Geometry Columns
We will define the core architecture here. For spatial speed, all geometries are wrapped up with `CreateSpatialIndex()` functions which internally manage an R-tree logic layout.

In [12]:
# Drop tables if exist
cursor.execute("DROP TABLE IF EXISTS cafes")
cursor.execute("DROP TABLE IF EXISTS pois")

print("Building tables...")

# CAFE TABLE
cursor.execute("""
    CREATE TABLE cafes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        lat REAL NOT NULL,
        lon REAL NOT NULL,
        revenue REAL,
        opening_year INTEGER,
        description TEXT,
        rank INTEGER,
        rating REAL,
        reviews_count INTEGER,
        weekly_hours TEXT,
        is_active BOOLEAN,
        is_verified BOOLEAN
    )
""")
# Add Geometry and Index (EPSG 4326 = standard GPS coordinates format)
cursor.execute("SELECT AddGeometryColumn('cafes', 'geometry', 4326, 'POINT', 'XY')")
cursor.execute("SELECT CreateSpatialIndex('cafes', 'geometry')")


# GENERIC POI TABLE (Banks, Education, Health...)
cursor.execute("""
    CREATE TABLE pois (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        poi_type TEXT NOT NULL,
        name TEXT,
        lat REAL NOT NULL,
        lon REAL NOT NULL,
        subtype TEXT,
        address TEXT,
        rank INTEGER,
        rating REAL,
        reviews_count INTEGER,
        weekly_hours TEXT,
        is_active BOOLEAN,
        is_verified BOOLEAN
    )
""")
cursor.execute("SELECT AddGeometryColumn('pois', 'geometry', 4326, 'POINT', 'XY')")
cursor.execute("SELECT CreateSpatialIndex('pois', 'geometry')")


# CAFE METRICS (Pre-computed edge connections for GNN mapping)
cursor.execute("""
    CREATE TABLE cafe_metrics (
        cafe_id INTEGER PRIMARY KEY,
        nearest_bank_dist REAL,
        nearest_school_dist REAL,
        poi_count_500m INTEGER,
        road_network_node_id INTEGER,
        FOREIGN KEY(cafe_id) REFERENCES cafes(id)
    )
""")

conn.commit()
print("✓ Schema definitions successful and geometry columns created!")

Building tables...


✓ Schema definitions successful and geometry columns created!


## 3. Extract & Load Process
Iterate through the existing target CSVs and inject them into their destination schemas, generating a `POINT()` mapping via SQLite’s geometries.

In [13]:
def map_csv_to_table(filename, table_name, poi_type=None):
    csv_file = CSV_DIR / filename
    if not csv_file.exists():
        print(f"Skipping {filename}: File not found.")
        return 0
        
    df = pd.read_csv(csv_file)
    count = 0
    
    # Helper function to safely extract numeric values
    def safe_numeric(val, dtype=float):
        if pd.isna(val) or val == '' or val == 'nan':
            return None
        try:
            return dtype(val)
        except (ValueError, TypeError):
            return None
    
    # Helper function to safely extract boolean values
    def safe_boolean(val):
        if pd.isna(val) or val == '':
            return None
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.lower() in ('true', '1', 'yes', 'y')
        return None
    
    for _, row in df.iterrows():
        lat = float(row.get('lat', 0))
        lon = float(row.get('lng', 0))
        name = row.get('name', 'Unknown')
        
        # Extract numeric and boolean columns
        rank = safe_numeric(row.get('rank'), int)
        rating = safe_numeric(row.get('rating'), float)
        reviews_count = safe_numeric(row.get('reviews_count'), int)
        weekly_hours = str(row.get('weekly_hours', '')) if pd.notna(row.get('weekly_hours')) else None
        is_active = safe_boolean(row.get('is_active'))
        is_verified = safe_boolean(row.get('is_verified'))
        
        if table_name == 'cafes':
            rev = safe_numeric(row.get('revenue'), float)
            year = safe_numeric(row.get('opening_year'), int)
            desc = row.get('description', '')
            
            cursor.execute("""
                INSERT INTO cafes (name, lat, lon, revenue, opening_year, description, 
                                   rank, rating, reviews_count, weekly_hours, is_active, is_verified) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (name, lat, lon, rev, year, desc, rank, rating, reviews_count, weekly_hours, is_active, is_verified))
            
            last_id = cursor.lastrowid
            # Populate Spatial Mapping Geometry
            cursor.execute(f"UPDATE cafes SET geometry = GeomFromText('POINT({lon} {lat})', 4326) WHERE id = {last_id}")
            
        else:
            subtype = row.get('category', '')
            address = row.get('address', '')
            
            cursor.execute("""
                INSERT INTO pois (poi_type, name, lat, lon, subtype, address, 
                                  rank, rating, reviews_count, weekly_hours, is_active, is_verified) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (poi_type, name, lat, lon, subtype, address, rank, rating, reviews_count, weekly_hours, is_active, is_verified))
            
            last_id = cursor.lastrowid
            # Populate Spatial Mapping Geometry
            cursor.execute(f"UPDATE pois SET geometry = GeomFromText('POINT({lon} {lat})', 4326) WHERE id = {last_id}")
            
        count += 1
        
    conn.commit()
    print(f"✓ Loaded {count} rows from {filename} -> `{table_name}`")
    return count

print("Migration in progress...")
# Migrate Cafes
map_csv_to_table('cafes.csv', 'cafes')

# Migrate POIs Contexts
for file_name, poi_flag in FILE_MAPPINGS.items():
    map_csv_to_table(file_name, 'pois', poi_type=poi_flag)
print("Migration completed.")

Migration in progress...
✓ Loaded 4052 rows from cafes.csv -> `cafes`
✓ Loaded 613 rows from banks.csv -> `pois`
✓ Loaded 1804 rows from education.csv -> `pois`
✓ Loaded 1575 rows from health.csv -> `pois`
✓ Loaded 2200 rows from temples.csv -> `pois`
✓ Loaded 1913 rows from other.csv -> `pois`
Migration completed.


## 4. Verification Check and Real-world SQL test
Execute a 500m radius check natively via SpatiaLite around the first mapped cafe to confirm R-Tree functionality.

In [14]:
print("--- DATABASE STATISTICS ---\n")
cursor.execute("SELECT COUNT(*) FROM cafes")
print(f"Total Cafes Loaded: {cursor.fetchone()[0]}")

cursor.execute("SELECT poi_type, COUNT(*) FROM pois GROUP BY poi_type")
print("\nPOI Distributions:")
for row in cursor.fetchall():
    print(f"  {row[0].capitalize()}: {row[1]}")

print("\n\n--- TESTING SPATIAL INDEX PERFORMANCE ---")

# Let's see the performance vs looping!
cursor.execute("SELECT id, name FROM cafes LIMIT 1")
cafe = cursor.fetchone()

if cafe:
    print(f"\nFinding POI density within 500m of Cafe #{cafe[0]} '{cafe[1]}':")
    
    # Notice how we let SQL perform the radius search directly using geometry points!
    # ST_Distance(geom1, geom2, 1) calculates in meters on the WGS84 ellipsoid
    cursor.execute("""
        SELECT poi_type, COUNT(*) as count 
        FROM pois 
        WHERE ST_Distance(geometry, (SELECT geometry FROM cafes WHERE id = ?), 1) < 500
        GROUP BY poi_type
    """, (cafe[0],))
    
    for row in cursor.fetchall():
        print(f"  Found {row[1]} {row[0].capitalize()}s within 500m.")
else:
    print("No cafes found to run test with.")
    
# Clean Exit
conn.close()

--- DATABASE STATISTICS ---

Total Cafes Loaded: 4052

POI Distributions:
  Bank: 613
  Hospital: 1575
  Other: 1913
  School: 1804
  Temple: 2200


--- TESTING SPATIAL INDEX PERFORMANCE ---

Finding POI density within 500m of Cafe #1 'Happy Hills Adventure':
  Found 1 Others within 500m.
  Found 1 Schools within 500m.
  Found 4 Temples within 500m.


In [15]:

# Verify new columns exist and display sample data
print("\n--- SAMPLE DATA WITH NEW COLUMNS ---\n")

print("Sample Cafes with new data:")
cursor.execute("""
    SELECT id, name, rating, reviews_count, rank, weekly_hours, is_active, is_verified 
    FROM cafes 
    LIMIT 3
""")
cafes_sample = cursor.fetchall()
if cafes_sample:
    for row in cafes_sample:
        print(f"  ID: {row[0]}, Name: {row[1]}, Rating: {row[2]}, Reviews: {row[3]}, Rank: {row[4]}")
        print(f"    Hours: {row[5]}, Active: {row[6]}, Verified: {row[7]}")
else:
    print("  No cafes found (columns created but may need data)")

print("\nSample POIs with new data:")
cursor.execute("""
    SELECT id, poi_type, name, rating, reviews_count, rank, weekly_hours, is_active 
    FROM pois 
    LIMIT 3
""")
pois_sample = cursor.fetchall()
if pois_sample:
    for row in pois_sample:
        print(f"  ID: {row[0]}, Type: {row[1]}, Name: {row[2]}, Rating: {row[3]}, Reviews: {row[4]}")
        print(f"    Rank: {row[5]}, Hours: {row[6]}, Active: {row[7]}")
else:
    print("  No POIs found (columns created but may need data)")

# Show columns available in tables
print("\n--- NEW COLUMNS AVAILABLE ---")
print("Cafes table columns: name, lat, lon, revenue, opening_year, description, rank, rating, reviews_count, weekly_hours, is_active, is_verified, geometry")
print("POIs table columns: poi_type, name, lat, lon, subtype, address, rank, rating, reviews_count, weekly_hours, is_active, is_verified, geometry")



--- SAMPLE DATA WITH NEW COLUMNS ---

Sample Cafes with new data:


ProgrammingError: Cannot operate on a closed database.